# Task 4 — Vowel Analysis in Python
### Erasmus Mundus Master in Linguistic Data Science — Lab 1

This notebook implements the full vowel analysis pipeline described in Task 4 of the lab guide. It uses **Parselmouth** (Python bindings for Praat's analysis engine) to extract acoustic features from sustained vowel recordings, and **scikit-learn** to build statistical models of the vowel space.

**Corpus:** Sustained productions of /a/, /i/, /u/ by male and female speakers.  
**File naming convention:** `vowel_<vowel>_<gender><id>.wav`  
(e.g. `vowel_a_m01.wav`, `vowel_i_f02.wav`)

---
**Pipeline overview:**
1. Environment setup
2. Silence removal + 250 ms steady-state trimming
3. Feature extraction: F0, F1, F2
4. Gaussian modelling of F1 per (vowel, gender)
5. F1/F2 scatter plot + Gaussian Mixture Model (K = 6) + BIC model selection
6. Report questions

## Step 1 — Environment Setup

Install `praat-parselmouth` and import all required libraries. Then mount Google Drive and point `AUDIO_DIR` to the folder containing your WAV files.

> **Note:** After running `!pip install`, restart the Colab runtime once before importing.

In [ ]:
# --- Install dependencies ---
!pip install praat-parselmouth --quiet

import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.linalg as la
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Ellipse
import parselmouth
from parselmouth.praat import call
from pathlib import Path
from tqdm.notebook import tqdm

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# ⚠️  Adjust this path to the folder where you uploaded the WAV files
AUDIO_DIR = '/content/drive/MyDrive/LinguisticDataScience_Lab/vowels/'

print("Libraries loaded. AUDIO_DIR =", AUDIO_DIR)

## Step 2 — Silence Removal and Steady-State Trimming

A sustained-vowel recording usually starts and ends with articulatory transients (CV onset coarticulation, phonation release) that introduce measurement bias. The preprocessing pipeline is:

1. **Voice Activity Detection (VAD):** A frame is classified as voiced if its Praat intensity satisfies:
   > *I*(t) > *I*_FS + θ_VAD,   θ_VAD = **−40 dBFS**

   where *I*_FS is the **theoretical full-scale intensity for a 16-bit WAV** in Praat's dB scale. Praat normalises 16-bit samples by dividing by 2¹⁵ = 32 768, so the maximum normalised amplitude is 1. Using Praat's intensity reference *I*₀ = 4×10⁻¹⁰ W/m²:

The function returns a `parselmouth.Sound` object representing only the steady-state core, or `None` when the voiced region is too short.

   > *I*_FS = 10 · log₁₀(1 / 4×10⁻¹⁰) ≈ **93.98 dB**

2. **Margin trimming:** discard the first and last **250 ms** of the voiced interval. LPC formant tracks typically stabilise within 100–200 ms of vowel onset (Hillenbrand et al., 1995), so 250 ms is a conservative safety margin.

   The effective threshold *I*_FS + θ_VAD ≈ 53.98 dB is therefore **fixed and recording-level independent** — it depends only on the bit depth, not on how loud an individual speaker was recorded. Pre-emphasis is applied internally by Praat's formant and pitch analysers; no explicit filtering step is needed here.

In [ ]:
# Full-scale intensity reference for 16-bit WAV in Praat's dB scale.
# Praat normalises samples to [-1, 1] by dividing by 2^15 = 32 768.
# I_FS = 10 * log10( (2^15 / 2^15)^2 / I_ref ) = 10 * log10(1 / 4e-10)
I_REF      = 4e-10                               # Praat intensity reference (W/m²)
PEAK_DB_FS = 10 * np.log10(1.0 / I_REF)         # ≈ 93.98 dB  (16-bit full scale)
print(f"Full-scale reference intensity: {PEAK_DB_FS:.2f} dB")


def extract_steady_state(sound, vad_dbfs=-40.0, margin_ms=250.0):
    """
    Removes leading/trailing silence and trims the first and last
    margin_ms milliseconds of the detected voiced interval.

    Parameters
    ----------
    sound      : parselmouth.Sound  (16-bit WAV loaded by Parselmouth)
    vad_dbfs   : VAD threshold in dBFS (full-scale), always negative.
                 A frame is voiced if I(t) > PEAK_DB_FS + vad_dbfs.
                 Default -40 dBFS -> threshold ~53.98 dB (Praat absolute scale).
    margin_ms  : milliseconds to discard from each end of the voiced region.

    Returns
    -------
    parselmouth.Sound of the steady-state vowel, or None if too short.
    """
    margin_s  = margin_ms / 1000.0
    intensity = call(sound, 'To Intensity', 100.0, 0.0, 'yes')
    n_frames  = call(intensity, 'Get number of frames')

    # Fixed absolute threshold (dB, Praat scale) — independent of recording level
    threshold_db = PEAK_DB_FS + vad_dbfs      # e.g. 93.98 + (-40) = 53.98 dB

    voiced_times = []
    for i in range(1, n_frames + 1):
        t   = call(intensity, 'Get time from frame number', i)
        val = call(intensity, 'Get value in frame', i)
        if val > threshold_db:
            voiced_times.append(t)

    if not voiced_times:
        return None                         # entirely silent

    t_start = voiced_times[0]  + margin_s
    t_end   = voiced_times[-1] - margin_s

    if t_start >= t_end:
        return None                         # voiced region shorter than 2×margin

    return call(sound, 'Extract part', t_start, t_end, 'rectangular', 1.0, 'yes')


## Step 3 — Feature Extraction: F0, F1, F2

### F0 — Praat autocorrelation pitch tracker

Praat estimates F0 frame-by-frame using normalised autocorrelation with Viterbi smoothing (Boersma, 1993). We report the **median over voiced frames**, which is robust to occasional octave errors.

The pitch ceiling is **gender-adapted** to reduce octave-error risk:
- Male speakers: ceiling = 300 Hz
- Female speakers: ceiling = 400 Hz

### F1/F2 — LPC Burg formant tracker

Formants are extracted via Praat's `To Formant (burg)`, which fits an all-pole AR(p) model using the Burg lattice algorithm. The key parameter is `maximum_formant`:
- Male: 5000 Hz (shorter expected formant range)
- Female: 5500 Hz (shorter vocal tract → higher formants)

A 25 ms analysis window and pre-emphasis from 50 Hz are applied. Per-frame F1 and F2 values are filtered by sanity bounds (F1 ∈ [200, 1200] Hz; F2 ∈ [500, 3500] Hz) before taking the median.

In [ ]:
def extract_f0(sound, gender):
    """Return median F0 (Hz) over voiced frames; NaN if no voiced frames found."""
    ceil = 300.0 if gender == 'm' else 400.0
    pitch = call(sound, "To Pitch", 0.0, 75.0, ceil)
    n = call(pitch, "Get number of frames")
    vals = []
    for i in range(1, n + 1):
        v = call(pitch, "Get value in frame", i, "Hertz")
        if v == v:   # NaN check
            vals.append(v)
    return float(np.median(vals)) if vals else float("nan")


def extract_formants(sound, gender):
    """Return (median_F1, median_F2) in Hz using LPC Burg tracker."""
    max_f = 5000.0 if gender == 'm' else 5500.0
    formant = call(sound, "To Formant (burg)", 0.0, 5, max_f, 0.025, 50.0)
    n = call(formant, "Get number of frames")
    f1_vals, f2_vals = [], []
    for i in range(1, n + 1):
        f1 = call(formant, "Get value at time", 1,
                  call(formant, "Get time from frame number", i),
                  "Hertz", "Linear")
        f2 = call(formant, "Get value at time", 2,
                  call(formant, "Get time from frame number", i),
                  "Hertz", "Linear")
        if f1 == f1 and 200 < f1 < 1200:
            f1_vals.append(f1)
        if f2 == f2 and 500 < f2 < 3500:
            f2_vals.append(f2)
    med_f1 = float(np.median(f1_vals)) if f1_vals else float("nan")
    med_f2 = float(np.median(f2_vals)) if f2_vals else float("nan")
    return med_f1, med_f2


# Quick test on the first file found
FILE_RE = re.compile(r'vowel_([aiu])_([mf])([0-9]+)[.]wav', re.IGNORECASE)
test_files = sorted(Path(AUDIO_DIR).glob("vowel_*.wav"))
if test_files:
    snd_test = parselmouth.Sound(str(test_files[0]))
    print(f"Test file: {test_files[0].name}")
    print(f"Sampling frequency: {int(snd_test.sampling_frequency)} Hz")
    snd_test = extract_steady_state(snd_test)
    fname = test_files[0].name
    m = FILE_RE.match(fname)
    if m:
        g_test = m.group(2)
        print(f"File : {fname}")
        print(f"F0   : {extract_f0(snd_test, g_test):.1f} Hz")
        f1t, f2t = extract_formants(snd_test, g_test)
        print(f"F1   : {f1t:.1f} Hz")
        print(f"F2   : {f2t:.1f} Hz")

### Batch processing all audio files

We scan `AUDIO_DIR` for files matching the naming convention `vowel_<vowel>_<gender><id>.wav`, extract features for each file, and collect them into a Pandas DataFrame for subsequent analysis.

In [ ]:
# --- Option to re-analyze or load from CSV ---
REANALYZE_PRAAT = True # @param {type:"boolean"}

if REANALYZE_PRAAT or not os.path.exists('vowel_features.csv'):
    print("Re-analyzing audio files using Praat...")
    FILE_RE = re.re.compile(r'vowel_([aiu])_([mf])([0-9]+)[.]wav', re.IGNORECASE)
    records = []

    wav_files = sorted(Path(AUDIO_DIR).glob("vowel_*.wav"))
    print(f"Found {len(wav_files)} audio files")

    for wav_path in tqdm(wav_files, desc="Processing audio files"):
        m = FILE_RE.match(wav_path.name)
        if not m:
            print(f"  Skipping (name mismatch): {wav_path.name}")
            continue
        vowel, gender, spk_id = m.group(1).lower(), m.group(2).lower(), int(m.group(3))
        try:
            snd = parselmouth.Sound(str(wav_path))
            snd_steady = extract_steady_state(snd)
            if snd_steady is None:
                # No steady state found, assign NaN to features
                f0, f1, f2 = float("nan"), float("nan"), float("nan")
            else:
                f0 = extract_f0(snd_steady, gender)
                f1, f2 = extract_formants(snd_steady, gender)

            records.append(dict(vowel=vowel, gender=gender, speaker=spk_id,
                                F0=f0, F1=f1, F2=f2))
        except Exception as exc:
            print(f"  Error processing {wav_path.name}: {exc}")

    df = pd.DataFrame(records)
    print(f"\nExtracted features for {len(df)} tokens")
    df.to_csv('vowel_features.csv', index=False)
    print("DataFrame saved to 'vowel_features.csv'")
else:
    print("Loading Praat features from 'vowel_features.csv'...")
    df = pd.read_csv('vowel_features.csv')

print(df.groupby(['vowel', 'gender'])[['F0', 'F1', 'F2']].describe().round(1))

In [ ]:
VOWELS  = ['a', 'i', 'u']
GENDERS = [('m', 'Male'), ('f', 'Female')]
COLORS  = {'a': '#e41a1c', 'i': '#377eb8', 'u': '#4daf4a'}

## Step 4 — Gaussian Modelling of F1

### Statistical model

For each (vowel, gender) pair we fit a univariate Gaussian to the F1 distribution:

$$F_1 \mid v, g \;\sim\; \mathcal{N}\!\left(\mu_{v,g},\;\sigma_{v,g}^2\right)$$

The maximum-likelihood estimates are simply the sample mean and standard deviation. The resulting PDFs show how well-separated—or overlapping—the vowel categories are in the F1 dimension alone.

The 6 subplots (3 vowels × 2 genders) are plotted as histograms with the fitted Gaussian PDF overlaid. A vertical dashed line marks the mean.

In [ ]:
from scipy.stats import norm

VOWELS  = ['a', 'i', 'u']
GENDERS = [('m', 'Male'), ('f', 'Female')]
COLORS  = {'a': '#e41a1c', 'i': '#377eb8', 'u': '#4daf4a'}

fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=False, sharey=False)
fig.suptitle("Gaussian model of F1 per vowel and gender", fontsize=14)

for row, (g_code, g_label) in enumerate(GENDERS):
    for col, vow in enumerate(VOWELS):
        ax = axes[row, col]
        data = df[(df.vowel == vow) & (df.gender == g_code)]['F1'].dropna().values
        if len(data) < 2:
            ax.set_title(f"/{vow}/ {g_label} — no data")
            continue
        mu, sigma = norm.fit(data)
        x = np.linspace(data.min() - 2 * sigma, data.max() + 2 * sigma, 300)
        ax.hist(data, bins='auto', density=True,
                color=COLORS[vow], alpha=0.45, edgecolor='white')
        ax.plot(x, norm.pdf(x, mu, sigma),
                color=COLORS[vow], linewidth=2.0)
        ax.axvline(mu, color='black', linestyle='--', linewidth=1.2,
                   label=f'μ={mu:.0f} Hz\nσ={sigma:.0f} Hz')
        ax.legend(fontsize=8)
        ax.set_title(f"/{vow}/ — {g_label}")
        ax.set_xlabel("F1 (Hz)")
        ax.set_ylabel("Density")

plt.tight_layout()
plt.savefig("f1_gaussian_models.png", dpi=150)
plt.show()
print("Saved: f1_gaussian_models.png")

## Step 5 — F1/F2 Scatter Plot and Gaussian Mixture Model

### Scatter plot (vowel space)

The F1/F2 scatter plot is the classic representation of vowel space used in phonetics. By convention:
- **F2** is plotted on the **x-axis, reversed** (high F2 → left)  
- **F1** is plotted on the **y-axis, reversed** (high F1 → bottom)

This orientation mirrors the position of the tongue: high front vowels (/i/) appear top-left, low back vowels (/a/) appear bottom-left, and back rounded vowels (/u/) appear top-right.

In [ ]:
# LPC method

MARKERS = {'m': 'o', 'f': '^'}

fig, ax = plt.subplots(figsize=(8, 6))

for vow in VOWELS:
    for g_code, g_label in GENDERS:
        sub = df[(df.vowel == vow) & (df.gender == g_code)].dropna(subset=['F1', 'F2'])
        ax.scatter(sub['F2'], sub['F1'],
                   color=COLORS[vow], marker=MARKERS[g_code],
                   s=60, alpha=0.7,
                   label=f"/{vow}/ {g_label}")

ax.invert_xaxis()
# ax.invert_yaxis() # Removed to make F1 go from low to high
ax.set_xlabel("F2 (Hz)")
ax.set_ylabel("F1 (Hz)")
ax.set_title("F1/F2 vowel space")
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig("f1f2_scatter.png", dpi=150)
plt.show()
print("Saved: f1f2_scatter.png")

### Gaussian Mixture Model (GMM) with K = 6

A GMM models the joint F1/F2 density as a weighted sum of Gaussians:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}\!\left(\mathbf{x};\,\boldsymbol{\mu}_k,\,\boldsymbol{\Sigma}_k\right), \qquad \sum_{k=1}^{K}\pi_k = 1$$

We choose **K = 6** because the data has exactly 3 vowels × 2 genders = 6 natural sub-populations. Each component is expected to capture one (vowel, gender) cluster.

**Implementation notes:**
- Features are standardised with `StandardScaler` before fitting (different Hz scales for F1 vs F2 would bias the EM algorithm).
- Full (unconstrained) covariance matrices are used so the model can capture elongated or tilted clusters.
- `n_init=10` reruns EM from different random initialisations to avoid local optima.
- Ellipses at 1σ and 2σ are back-transformed to Hz for plotting.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Ellipse
import scipy.linalg as la

# ── Prepare data ─────────────────────────────────────────────────────────────
xy_df = df[['F1', 'F2']].dropna()
X_hz  = xy_df.values          # shape (N, 2)

scaler = StandardScaler()
X_std  = scaler.fit_transform(X_hz)

# ── Fit GMM with K = 6 ───────────────────────────────────────────────────────
gmm = GaussianMixture(n_components=6, covariance_type='full',
                      n_init=10, random_state=42)
gmm.fit(X_std)
print(f"GMM converged: {gmm.converged_}   |   log-likelihood: {gmm.lower_bound_:.3f}")

# ── Plot scatter + ellipses ──────────────────────────────────────────────────
def draw_ellipse(ax, mean_hz, cov_hz, n_std, **kw):
    """Draw a confidence ellipse for a 2-D Gaussian."""
    vals, vecs = la.eigh(cov_hz)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(np.abs(vals))
    ell = Ellipse(xy=mean_hz, width=w, height=h, angle=angle, **kw)
    ax.add_patch(ell)

fig, ax = plt.subplots(figsize=(9, 7))

# Scatter
for vow in VOWELS:
    for g_code, g_label in GENDERS:
        sub = df[(df.vowel == vow) & (df.gender == g_code)].dropna(subset=['F1','F2'])
        ax.scatter(sub['F2'], sub['F1'],
                   color=COLORS[vow], marker=MARKERS[g_code],
                   s=50, alpha=0.55, label=f"/{vow}/ {g_label}")

# Ellipses — back-transform means and covariances to Hz
S_inv = np.diag(scaler.scale_)          # scale_ gives std per feature
for k in range(gmm.n_components):
    mean_hz = scaler.inverse_transform(gmm.means_[k].reshape(1, -1)).ravel()
    cov_hz  = S_inv @ gmm.covariances_[k] @ S_inv
    # Note: F1 is dim-0, F2 is dim-1 → swap for (F2, F1) axes
    mean_plot = np.array([mean_hz[1], mean_hz[0]])
    cov_plot  = np.array([[cov_hz[1,1], cov_hz[1,0]],
                          [cov_hz[0,1], cov_hz[0,0]]])
    draw_ellipse(ax, mean_plot, cov_plot, n_std=1,
                 edgecolor='black', facecolor='none', linewidth=1.5)
    draw_ellipse(ax, mean_plot, cov_plot, n_std=2,
                 edgecolor='black', facecolor='none', linewidth=0.8,
                 linestyle='--')
    ax.plot(*mean_plot, 'k+', ms=8)

ax.invert_xaxis()
# ax.invert_yaxis()
ax.set_xlabel("F2 (Hz)")
ax.set_ylabel("F1 (Hz)")
ax.set_title("F1/F2 vowel space — GMM K=6 (1σ and 2σ ellipses)")
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig("f1f2_gmm.png", dpi=150)
plt.show()
print("Saved: f1f2_gmm.png")

### BIC model selection — choosing K

The **Bayesian Information Criterion** penalises model complexity:

$$\text{BIC}(K) = -2\ln\hat{\mathcal{L}} + p_K \ln N$$

where $p_K$ is the number of free parameters and $N$ is the number of training samples. The model that minimises BIC achieves the best trade-off between goodness-of-fit and parsimony. We sweep K from 2 to 10 and plot the BIC curve.

In [ ]:
K_range = range(2, 11)
bic_scores = []

for k in K_range:
    g = GaussianMixture(n_components=k, covariance_type='full',
                        n_init=10, random_state=42)
    g.fit(X_std)
    bic_scores.append(g.bic(X_std))

best_k = list(K_range)[int(np.argmin(bic_scores))]
print(f"Best K by BIC: {best_k}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), bic_scores, marker='o', linewidth=2)
ax.axvline(best_k, color='red', linestyle='--',
           label=f"Best K = {best_k}")
ax.set_xlabel("Number of components K")
ax.set_ylabel("BIC")
ax.set_title("BIC model selection for GMM")
ax.legend()
plt.tight_layout()
plt.savefig("gmm_bic.png", dpi=150)
plt.show()
print("Saved: gmm_bic.png")

---
## Lab Report Questions

Answer each question in a new cell below it. You may mix Markdown (for written answers) and Python (for code/figures).

### Q1 — VAD and segment duration

Report the mean and standard deviation of the duration (in ms) of the steady-state segments extracted by `extract_steady_state`. How does the duration vary across vowels and genders? What would happen to the F0/F1/F2 estimates if you skipped the VAD step?

*Write your answer here.*

### Q2 — F0 gender differences

Compare the mean F0 of male and female speakers for each vowel. Is the difference statistically significant? Run a two-sample t-test and report the t-statistic and p-value.

In [ ]:
from scipy.stats import ttest_ind

print(f"{'Vowel':<8} {'t-stat':>10} {'p-value':>12} {'Significant (p<0.05)?':>22}")
print("-" * 56)
for vow in VOWELS:
    male_f0   = df[(df.vowel == vow) & (df.gender == 'm')]['F0'].dropna()
    female_f0 = df[(df.vowel == vow) & (df.gender == 'f')]['F0'].dropna()
    if len(male_f0) > 1 and len(female_f0) > 1:
        t, p = ttest_ind(male_f0, female_f0, equal_var=False)
        sig = "Yes" if p < 0.05 else "No"
        print(f"/{vow}/     {t:>10.3f} {p:>12.4f} {sig:>22}")
    else:
        print(f"/{vow}/     insufficient data")

*Write your interpretation here.*

---

### Q3 — Gaussian F1 model: goodness of fit

For each (vowel, gender) pair, perform a Shapiro-Wilk normality test on the F1 values. Report the W-statistic and p-value. Does the Gaussian assumption hold?

In [ ]:
from scipy.stats import shapiro

print(f"{'Vowel':<6} {'Gender':<8} {'n':>4} {'W':>8} {'p-value':>10} {'Normal?':>10}")
print("-" * 52)
for vow in VOWELS:
    for g_code, g_label in GENDERS:
        data = df[(df.vowel == vow) & (df.gender == g_code)]['F1'].dropna().values
        if len(data) >= 3:
            W, p = shapiro(data)
            normal = "Yes" if p > 0.05 else "No"
            print(f"/{vow}/    {g_label:<8} {len(data):>4} {W:>8.4f} {p:>10.4f} {normal:>10}")
        else:
            print(f"/{vow}/    {g_label:<8} n={len(data)} — too few samples")

*Write your interpretation here.*

---

### Q4 — Vowel separability: Bhattacharyya coefficient

Compute the Bhattacharyya coefficient between each pair of vowels (male speakers only) using the fitted Gaussian F1 models. Which pair of vowels is most confusable in F1?

The Bhattacharyya coefficient between two univariate Gaussians is:

$$B(\mathcal{N}_1, \mathcal{N}_2) = \exp\!\left(-\frac{1}{4}\frac{(\mu_1-\mu_2)^2}{\sigma_1^2+\sigma_2^2}\right) \cdot \sqrt{\frac{2\sigma_1\sigma_2}{\sigma_1^2+\sigma_2^2}}$$

In [ ]:
from itertools import combinations

def bhattacharyya(mu1, s1, mu2, s2):
    var_sum = s1**2 + s2**2
    term1 = np.exp(-0.25 * (mu1 - mu2)**2 / var_sum)
    term2 = np.sqrt(2 * s1 * s2 / var_sum)
    return term1 * term2

params = {}
for vow in VOWELS:
    data = df[(df.vowel == vow) & (df.gender == 'm')]['F1'].dropna().values
    if len(data) >= 2:
        params[vow] = norm.fit(data)   # (mu, sigma)

print(f"{'Pair':<10} {'B coefficient':>16}  (closer to 1 → more overlap)")
print("-" * 48)
for v1, v2 in combinations(VOWELS, 2):
    if v1 in params and v2 in params:
        mu1, s1 = params[v1]
        mu2, s2 = params[v2]
        B = bhattacharyya(mu1, s1, mu2, s2)
        print(f"/{v1}/ vs /{v2}/  {B:>16.4f}")

*Write your interpretation here.*

---

### Q5 — LPC order sensitivity

Re-run the formant extraction with `n_formants` set to 4 and 6 (instead of 5). How do the median F1 and F2 values change? What is the recommended rule of thumb for choosing the LPC order?

In [ ]:
def extract_formants_nf(sound, gender, n_formants):
    """Like extract_formants but with variable n_formants."""
    max_f = 5000.0 if gender == 'm' else 5500.0
    if sound is None:
        return float("nan"), float("nan")
    formant = call(sound, "To Formant (burg)", 0.0, n_formants, max_f, 0.025, 50.0)
    n = call(formant, "Get number of frames")
    f1_vals, f2_vals = [], []
    for i in range(1, n + 1):
        t = call(formant, "Get time from frame number", i)
        f1 = call(formant, "Get value at time", 1, t, "Hertz", "Linear")
        f2 = call(formant, "Get value at time", 2, t, "Hertz", "Linear")
        if f1 == f1 and 200 < f1 < 1200:
            f1_vals.append(f1)
        if f2 == f2 and 500 < f2 < 3500:
            f2_vals.append(f2)
    return (float(np.median(f1_vals)) if f1_vals else float("nan"),
            float(np.median(f2_vals)) if f2_vals else float("nan"))

print(f"{'n_formants':<12} {'Vowel':<8} {'Gender':<8} {'F1 (Hz)':>10} {'F2 (Hz)':>10}")
print("-" * 52)
for nf in [4, 5, 6]:
    rows = []
    for wav_path in tqdm(sorted(Path(AUDIO_DIR).glob("vowel_*.wav")), desc=f"Processing with n_formants={nf}"):
        mm = FILE_RE.match(wav_path.name)
        if not mm:
            continue
        vow, gen = mm.group(1).lower(), mm.group(2).lower()
        snd = extract_steady_state(parselmouth.Sound(str(wav_path)))
        f1, f2 = extract_formants_nf(snd, gen, nf)
        rows.append(dict(vowel=vow, gender=gen, F1=f1, F2=f2))
    tmp = pd.DataFrame(rows)
    for vow in VOWELS:
        for g_code in ['m', 'f']:
            sub = tmp[(tmp.vowel == vow) & (tmp.gender == g_code)][['F1','F2']].mean()
            print(f"{nf:<12} /{vow}/    {g_code:<8} {sub.F1:>10.1f} {sub.F2:>10.1f}")

*Write your interpretation here.*

---

### Q6 — GMM component assignment

Label each GMM component (1–6) with its most likely (vowel, gender) identity by comparing the component mean to the per-group centroids in F1/F2. What does the confusion matrix between GMM assignment and true label look like?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Build true label strings
df_valid = df[['vowel', 'gender', 'F1', 'F2']].dropna().copy()
df_valid['true_label'] = df_valid['vowel'] + '_' + df_valid['gender']

X_pred  = scaler.transform(df_valid[['F1', 'F2']].values)
pred_k  = gmm.predict(X_pred)

# Map each component to nearest group centroid
label_order = [v + '_' + g for v in VOWELS for g in ['m', 'f']]
centroids_hz = {
    lbl: df_valid[df_valid.true_label == lbl][['F1', 'F2']].mean().values
    for lbl in label_order
}
centroids_std = {lbl: scaler.transform(c.reshape(1,-1)).ravel()
                 for lbl, c in centroids_hz.items() if not np.any(np.isnan(c))}

component_label = {}
for k in range(gmm.n_components):
    mu_k = gmm.means_[k]
    nearest = min(centroids_std,
                  key=lambda lbl: np.linalg.norm(mu_k - centroids_std[lbl]))
    component_label[k] = nearest

pred_label = [component_label[k] for k in pred_k]
true_label = df_valid['true_label'].tolist()

labels_present = sorted(set(true_label) | set(pred_label))
cm = confusion_matrix(true_label, pred_label, labels=labels_present)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_present)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title("GMM assignment vs true (vowel, gender) label")
plt.tight_layout()
plt.savefig("gmm_confusion.png", dpi=150)
plt.show()

*Write your interpretation here.*

---

### Q7 — BIC and linguistic prior

The BIC curve selects the statistically optimal K. Does it agree with the linguistically motivated K = 6? If BIC selects a different K, what might explain the discrepancy? Discuss the trade-off between model complexity and linguistic interpretability.

*Write your answer here.*

---
*End of notebook — Task 4, Erasmus Mundus Master in Linguistic Data Science, 2025–26*